In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pymargins import Margins, SurveyDesign

rng = np.random.default_rng(42)

# ---------------------------------------------------------------------------
# Simulate a stratified two-stage sample
# ---------------------------------------------------------------------------
# Population: 20 cities per stratum. We sample 3 cities per stratum
# (PSU = city) and then 100 people per sampled city.
strata_names = ["small", "medium", "large"]
n_strata = len(strata_names)
n_psu_per_stratum = 3
n_per_psu = 100

rows = []
for h, stratum in enumerate(strata_names):
    # City-level intercept (PSU effect)
    city_effects = rng.normal(0, 0.3, 20)
    sampled_cities = rng.choice(20, size=n_psu_per_stratum, replace=False)
    for psu_idx, city in enumerate(sampled_cities):
        # Individual-level data
        age = rng.integers(25, 70, n_per_psu)
        income = rng.normal(50 + 5 * h, 15, n_per_psu)
        # True DGP: older, richer, and larger-city people are more likely
        lp = (-2.0
              + 0.04 * age
              + 0.02 * income
              + 0.3 * h
              + city_effects[city])
        prob = 1 / (1 + np.exp(-lp))
        y = rng.binomial(1, prob)
        # Sampling weight varies by stratum (oversampling small cities)
        weight = (20 / 3) * (500 / 100) * (1 + 0.3 * h)
        rows.append(pd.DataFrame({
            "y": y,
            "age": age,
            "income": income,
            "stratum": stratum,
            "psu": f"{stratum}_{psu_idx}",
            "weight": weight,
        }))

df = pd.concat(rows, ignore_index=True)
print(df.head())
print(f"\nN = {len(df)}, strata = {df['stratum'].unique().tolist()}")
print(f"PSUs per stratum = {df.groupby('stratum')['psu'].nunique().tolist()}")

   y  age     income stratum      psu     weight
0  1   68  33.000692   small  small_0  33.333333
1  1   45  36.208216   small  small_0  33.333333
2  1   65  57.457411   small  small_0  33.333333
3  0   55  52.136386   small  small_0  33.333333
4  0   60  60.357280   small  small_0  33.333333

N = 900, strata = ['small', 'medium', 'large']
PSUs per stratum = [3, 3, 3]


In [2]:
fit = smf.glm(
    "y ~ age + income",
    data=df,
    family=sm.families.Binomial(),
    freq_weights=df["weight"].values,
).fit(disp=False)
print(fit.summary().tables[1])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.2648      0.064    -19.695      0.000      -1.391      -1.139
age            0.0360      0.001     35.579      0.000       0.034       0.038
income         0.0166      0.001     20.396      0.000       0.015       0.018


In [3]:
m_naive = Margins(fit, at="overall")
m_weighted = Margins(fit, at="overall", weights=df["weight"].values)

print("Naive  P(y=1 | age=40):",
      float(m_naive.predict(atexog={"age": [40]}).estimate))
print("Weighted P(y=1 | age=40):",
      float(m_weighted.predict(atexog={"age": [40]}).estimate))

Naive  P(y=1 | age=40): 0.7431016920318492


Weighted P(y=1 | age=40): 0.7455762100563835


In [4]:
print("Naive  AME of age:")
print(m_naive.dydx("age").summary())
print("\nWeighted AME of age:")
print(m_weighted.dydx("age").summary())

Naive  AME of age:


           Margins Result (delta, level=0.95)           
     estimate  std err        z  P>|z|  [95% Conf. Int.]
--------------------------------------------------------
age    0.0059   0.0002  37.1885  0.000    0.0055, 0.0062

n = 900
κ: 0.008
Delta-vs-sim disagreement: 0.027%

Weighted AME of age:
           Margins Result (delta, level=0.95)           
     estimate  std err        z  P>|z|  [95% Conf. Int.]
--------------------------------------------------------
age    0.0058   0.0002  37.0290  0.000    0.0055, 0.0061

n = 900
κ: 0.008
Delta-vs-sim disagreement: 0.156%


In [5]:
# Map stratum names to integer codes for SurveyDesign
stratum_codes = df["stratum"].astype("category").cat.codes

survey = SurveyDesign(
    weights=df["weight"].values,
    psu=df["psu"].values,
    strata=stratum_codes.values,
)

m_survey = Margins(fit, survey_design=survey, weights=df["weight"].values)
print(m_survey.dydx("age").summary())

           Margins Result (delta, level=0.95)          
     estimate  std err       z  P>|z|  [95% Conf. Int.]
-------------------------------------------------------
age    0.0058   0.0012  4.9503  0.000    0.0035, 0.0081

n = 900
κ: 0.052
Delta-vs-sim disagreement: 3.533%


In [6]:
fit_unweighted = smf.glm(
    "y ~ age + income",
    data=df,
    family=sm.families.Binomial(),
).fit(disp=False)

m_posthoc = Margins(
    fit_unweighted,
    survey_design=survey,
    weights=df["weight"].values,
)
print(m_posthoc.dydx("age").summary())

           Margins Result (delta, level=0.95)          
     estimate  std err       z  P>|z|  [95% Conf. Int.]
-------------------------------------------------------
age    0.0058   0.0012  4.8358  0.000    0.0035, 0.0082

n = 900
κ: 0.049
Delta-vs-sim disagreement: 1.338%


In [7]:
m_boot = Margins(
    fit,
    survey_design=survey,
    weights=df["weight"].values,
    method="bootstrap",
    n_boot=300,
    rng_seed=42,
)
print(m_boot.dydx("age").summary())

          Margins Result (bootstrap, level=0.95)          
     estimate  std err  statistic  P>|z|  [95% Conf. Int.]
----------------------------------------------------------
age    0.0058   0.0010     0.0058  0.000    0.0040, 0.0078

n = 900
κ: 0.052


In [8]:
results = pd.DataFrame({
    "approach": ["naïve", "weighted only", "survey linearization",
                 "survey bootstrap", "post-hoc unweighted"],
    "estimate": [
        float(m_naive.dydx("age").estimate),
        float(m_weighted.dydx("age").estimate),
        float(m_survey.dydx("age").estimate),
        float(m_boot.dydx("age").estimate),
        float(m_posthoc.dydx("age").estimate),
    ],
    "std_error": [
        float(m_naive.dydx("age").std_error),
        float(m_weighted.dydx("age").std_error),
        float(m_survey.dydx("age").std_error),
        float(m_boot.dydx("age").std_error),
        float(m_posthoc.dydx("age").std_error),
    ],
})
print(results.round(5))

               approach  estimate  std_error
0                 naïve   0.00586    0.00016
1         weighted only   0.00583    0.00016
2  survey linearization   0.00583    0.00118
3      survey bootstrap   0.00583    0.00104
4   post-hoc unweighted   0.00583    0.00121
